# SGD vs Adam

En esta notebook vamos a comparar **SGD** y **Adam** en escenarios simples y controlados.
La idea es observar cómo influyen factores como el *learning rate*, el *momentum* y la presencia de *ruido* en el gradiente.

## Utils

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración de estilo seaborn
sns.set(style="whitegrid")

In [ ]:
def plot_function_and_grad(f, df, x_range=(-3, 3), title=""):
  xs = np.linspace(*x_range, 200)
  fig, axes = plt.subplots(1, 2, figsize=(12,4))

  # Función
  axes[0].plot(xs, f(xs), label="f(x)", color="C0")
  axes[0].axhline(0, color='k', linewidth=0.5)
  axes[0].set_title("Función")
  axes[0].legend()

  # Derivada
  axes[1].plot(xs, df(xs), label="f'(x)", color="C1")
  axes[1].axhline(0, color='k', linewidth=0.5)
  axes[1].set_title("Derivada")
  axes[1].legend()

  fig.suptitle(title)
  plt.show()

def run_optimizer(f, df, opt, x0=2.5, steps=50):
  x = np.array(x0)
  history = [(0, x, f(x))]
  for t in range(steps):
    g = df(x)
    x = opt.update(x, g)
    history.append((t+1, x, f(x)))
  return np.array(history)

def plot_trajectories(histories, title=""):
  fig, ax = plt.subplots(figsize=(7,4))

  for label, h in histories.items():
    sns.lineplot(x=h[:,0], y=h[:,1], label=label, ax=ax, marker="o", markersize=4)

  ax.axhline(0, color="k", linewidth=0.8, linestyle="--")
  ax.set_xlabel("Step")
  ax.set_ylabel("x value")
  ax.set_title(title)
  ax.legend()
  plt.show()

## Optimizadores

Para el entrenamiento de una red neuronal usamos **algoritmos de optimización** que ajustan los parámetros siguiendo el gradiente. Existen muchas variantes de optimizadores, para este ejemplo vamos a utilizar **SGD** y **Adam**

---
### 1. SGD (Stochastic Gradient Descent)
Actualizar cada parámetro en la dirección contraria a su gradiente, escalado por una tasa de aprendizaje.

$$
\theta_{t+1} = \theta_t - \alpha\ \cdot\ g_t  
$$

- $\theta_t$: parámetros de la red en el paso $t$.
- $\alpha$: *learning rate*
- $g_t$: gradiente respecto a la Loss

Cuando trabajamos con batchs (o mini-batchs), el gradiente se calcula solo con un lote de datos, lo que introduce ruido, de ahi el **Stochastic**.

**Variante con momentum**

Añadir una media móvil de gradientes:
$$
v_t = \beta v_{t-1} + g_t \\
\theta_{t+1} = \theta_t - \alpha v_t
$$

---
### 2. Adam (Adaptive Moment Estimation)
Mantiene promedios móviles de gradientes y sus cuadrados:
- $m_t$: promedio de gradientes (momento de primer orden)
- $v_t$: promedio de gradientes al cuadrado (momento de segundo orden)

Fórmulas:
1. Actualización de momentos:
$$
m_t = \beta_1 m_{t-1} + (1-\beta)g_t \\
v_t = \beta_2 v_{t-1} + (1-\beta)g_t^2
$$

1. Corrección por sesgo (importante en pasos iniciales):
$$
\hat{m}_t = \frac{m_t}{1-\beta_1^t},\ \ \hat{v}_t = \frac{v_t}{1-\beta_2^t},
$$

1. Actualización de parámetros:
$$
\theta_{t+1} = \theta_t - \alpha \cdot \frac{ \hat{m}_t}{\sqrt{\hat{v}_t} + ϵ}
$$

- $g_t$: gradiente en el paso $t$
- $\beta_1 ≈ 0.9, \beta_2 ≈ 0.999$
- $ϵ ≈ 1e^{-08}$: pequeño valor para evitar dividir por cero

In [ ]:
class SGD:
    def __init__(self, lr=0.1, momentum=0.0):
        self.lr = lr
        self.momentum = momentum
        self.v = 0.0

    def update(self, x, g):
        self.v = self.momentum * self.v + g
        return x - (self.lr * self.v)


class Adam:
    def __init__(self, lr=0.1, betas=(0.9, 0.999), eps=1e-8):
        self.lr = lr
        self.beta1, self.beta2 = betas
        self.eps = eps
        self.m = 0.0
        self.v = 0.0
        self.t = 0

    def update(self, x, g):
        self.t += 1
        self.m = self.beta1 * self.m + (1 - self.beta1) * g
        self.v = self.beta2 * self.v + (1 - self.beta2) * (g**2)

        # Corrección de sesgo
        m_hat = self.m / (1 - self.beta1**self.t)
        v_hat = self.v / (1 - self.beta2**self.t)

        return x - self.lr * m_hat / (np.sqrt(v_hat) + self.eps)

---
## Función cuadrática $f(x) = x^2$

Comenzamos con la función más simple: una parábola. Aquí el mínimo está en $x=0$ y el gradiente es lineal.

In [ ]:
def f1(x):
    return x**2

def f1grad(x):
    return 2 * x

In [ ]:
plot_function_and_grad(f1, f1grad, title="f(x) = x^2")

### Efecto del learning rate

Al optimizar con SGD y Adam veremos cómo el *learning rate* cambia la velocidad y estabilidad de la convergencia.
- *Learning rate* muy bajo $\rightarrow$ convergencia lenta
- *learning rate* muy alto $\rightarrow$ oscilaciones o divergencia

In [ ]:
# Diferentes learning rates
learning_rates = [0.01, 0.1, 0.5]

# SGD con distintos LR
histories_sgd = {}
for lr in learning_rates:
  opt = SGD(lr=lr)
  h = run_optimizer(f1, f1grad, opt, steps=100)
  histories_sgd[f"SGD lr={lr}"] = h

plot_trajectories(histories_sgd, title="SGD con distintos learning rates (f(x)=x^2)")

In [ ]:
# Adam con distintos LR
histories_adam = {}
for lr in learning_rates:
  opt = Adam(lr=lr)
  h = run_optimizer(f1, f1grad, opt, steps=100)
  histories_adam[f"Adam lr={lr}"] = h


plot_trajectories(histories_adam, title="Adam con distintos learning rates (f(x)=x^2)")

## Efecto del *Momentum* en SGD

El momentum en SGD agrega una memoria de los gradientes pasados, lo que genera una efecto de **aceleración**.


In [ ]:
# Diferentes configuraciones de momentum
opt_sgd_no_mom   = SGD(lr=0.1, momentum=0.0)
opt_sgd_mom_05   = SGD(lr=0.1, momentum=0.5)
opt_sgd_mom_09   = SGD(lr=0.1, momentum=0.9)

h_no_mom   = run_optimizer(f1, f1grad, opt_sgd_no_mom)
h_mom_05   = run_optimizer(f1, f1grad, opt_sgd_mom_05)
h_mom_09   = run_optimizer(f1, f1grad, opt_sgd_mom_09)

plot_trajectories({
    "SGD (mom=0.0)"  : h_no_mom,
    "SGD (mom=0.5)"  : h_mom_05,
    "SGD (mom=0.9)"  : h_mom_09,
}, title="Efecto del Momentum")


## Función con ruido  

Cómo observamos en las secciones anteriores, pareciera que SGD con poco (o sin) momentum es la mejor opción. Pero, ¿qué pasa si tenemos ruido en el gradiente? Esto es lo que normalmente pasa al trabajar con **mini-batches** asi que vamos a simularlo.

Ahora usamos una función un poco más compleja:  

$$
f(x) = x^3 + 0.5x^2
$$

**Agregar ruido**
Esto hace que el optimizador deba lidiar con señales inestables, deberíamos ver que:  

- **SGD clásico** se vuelve muy irregular.  
- **SGD con momentum** mejora la estabilidad, pero sigue sensible al ruido.  
- **Adam** se adapta mejor y avanza de manera más consistente.  


In [ ]:
f2 = lambda x: x**3 + 0.5*x**2
f2grad = lambda x: 3*x**2 + x

plot_function_and_grad(f2, f2grad, x_range=(-5, 5), title="f(x) = x^3 + 0.5x^2")

In [ ]:
def f2grad_noisy(x, noise_scale=10.0):
    return f2grad(x) + np.random.normal(scale=noise_scale, size=x.shape)

plot_function_and_grad(f2, f2grad_noisy, x_range=(-5, 5), title="f(x) = x^3 + 0.5x^2 con ruido")

In [ ]:
opt_sgd_noise = SGD(lr=0.01, momentum=0.0)
opt_mom_noise = SGD(lr=0.01, momentum=0.2)
opt_adam_noise = Adam(lr=0.1)

steps = 100
x0 = 5.0
h_sgd_noise = run_optimizer(f2, f2grad_noisy, opt_sgd_noise, steps=steps, x0=x0)
h_mom_noise = run_optimizer(f2, f2grad_noisy, opt_mom_noise, steps=steps, x0=x0)
h_adam_noise = run_optimizer(f2, f2grad_noisy, opt_adam_noise, steps=steps, x0=x0)

plot_trajectories({"SGD": h_sgd_noise,
                   "SGD (momentum)": h_mom_noise,
                   "Adam": h_adam_noise},
                  title="Optimización con ruido (f(x)=x^3+0.5x^2)")

## Función con 2 dimensiones

Como vimos, Adam es más estable ante el ruido, sin embargo, SGD pareciera seguir convergiendo más rapido. Ahora agreguemos una capa de complejidad más, ¿Qué pasa si nuestra función tiene 2 parámetros en lugar de uno solo?

Ahora vamos a usar la función:
$$
f(x_1, x_2) = \sin(x_1) * \cos(x_2) + \sin(0.5x_1) * \cos(0.5x_2)
$$

Cuya derivadas son:
$$
\frac{\partial f}{\partial x_1} = \cos(x_1)*\cos(x_2) + 0.5 \cos(0.5 x_1)*\cos(0.5 x_2)\\
\frac{\partial f}{\partial x_2} = -\sin(x_1)*\sin(x_2) - 0.5 \sin(0.5 x_1)*\sin(0.5 x_2)
$$

In [ ]:
def f(x):
  x1 = x[:, 0]
  x2 = x[:, 1]
  return np.sin(x1) * np.cos(x2) + np.sin(0.5 * x1) * np.cos(0.5 * x2)

def grad_f(x):
  x1 = x[:, 0]
  x2 = x[:, 1]

  df_dx1 = np.cos(x1) * np.cos(x2) + 0.5 * np.cos(0.5 * x1) * np.cos(0.5 * x2)
  df_dx2 = -np.sin(x1) * np.sin(x2) - 0.5 * np.sin(0.5 * x1) * np.sin(0.5 * x2)

  return np.stack((df_dx1, df_dx2), axis=1)

def grad_noisy_f(x, noise_scale=5.0):
  return grad_f(x) + np.random.normal(scale=noise_scale, size=x.shape)

### Optimizadores para 2 dimensiones

Vamos a reimplementar los optimizadores **SGD** y **Adam** para trabajar ahora con arrays de números, en lugar de escalares.

In [ ]:
class SGD:
    def __init__(self, lr=0.1, momentum=0.0):
        self.lr = lr
        self.momentum = momentum
        self.v = None

    def update(self, x, g):
        # Inicializa la velocidad como un array de ceros
        # la primera vez que se llama a la función
        if self.v is None:
            self.v = np.zeros_like(x)

        self.v = self.momentum * self.v + g
        return x - self.lr * self.v

class Adam:
    def __init__(self, lr=0.1, betas=(0.9, 0.999), eps=1e-8):
        self.lr = lr
        self.beta1, self.beta2 = betas
        self.eps = eps
        self.m = None  # Se inicializará como un array
        self.v = None  # Se inicializará como un array
        self.t = 0

    def update(self, x, g):
        # Inicializa los arrays m y v con la forma correcta en la primera llamada
        if self.m is None:
            self.m = np.zeros_like(g)
            self.v = np.zeros_like(g)

        self.t += 1

        self.m = self.beta1 * self.m + (1 - self.beta1) * g
        self.v = self.beta2 * self.v + (1 - self.beta2) * (g**2)

        # Corrección de sesgo
        m_hat = self.m / (1 - self.beta1**self.t)
        v_hat = self.v / (1 - self.beta2**self.t)

        # Paso de actualización de los parámetros
        return x - self.lr * m_hat / (np.sqrt(v_hat) + self.eps)

### Utils

In [ ]:
def plot_2d_function(all_paths, f, title="Descenso de gradiente desde Múltiples Puntos de Partida"):
  plt.style.use('seaborn-v0_8-whitegrid') # Estilo visual
  plt.rcParams['font.size'] = 14
  plt.rcParams['axes.labelsize'] = 16
  plt.rcParams['axes.titlesize'] = 18

  fig, ax = plt.subplots(figsize=(8, 5), dpi=100)

  # Crear el grid para la función de contorno
  x1_vals = np.linspace(-2, 12, 100)
  x2_vals = np.linspace(-2, 12, 100)
  X1, X2 = np.meshgrid(x1_vals, x2_vals)
  Z = f(np.stack([X1.ravel(), X2.ravel()], axis=1)).reshape(X1.shape)

  # Graficar el contorno y la barra de colores
  contourf = ax.contourf(X1, X2, Z, levels=50, cmap='viridis')
  cbar = fig.colorbar(contourf, ax=ax)
  cbar.set_label('Valor de la función f(x, y)', rotation=270, labelpad=20)

  # Graficar las trayectorias del descenso
  for path in all_paths:
      # La trayectoria completa
      ax.plot(path[:, 0], path[:, 1], alpha=0.8, linewidth=1.5, zorder=2, linestyle="-.")
      # Puntos de inicio y final
      ax.scatter(path[0, 0], path[0, 1], c='lime', edgecolors='black', s=100, zorder=3, label="Punto de incio")
      ax.scatter(path[-1, 0], path[-1, 1], c='black', marker='*', s=100, linewidths=2, zorder=3, label="Punto final")

  handles, labels = ax.get_legend_handles_labels()
  unique_labels = list(dict.fromkeys(labels))
  unique_handles = [handles[labels.index(label)] for label in unique_labels]
  ax.legend(unique_handles, unique_labels, loc='upper right')

  # Configurar el gráfico
  ax.set_title(title)
  ax.set_xlabel('$x_1$')
  ax.set_ylabel('$x_2$')
  plt.tight_layout()
  plt.show()

In [ ]:
np.random.seed(42)  # Para reproducibilidad
start_points = np.random.uniform(0, 10, size=(10, 2))

### SGD
- Sin ruido
- Con ruido

In [ ]:
# Lista para almacenar las trayectorias de cada optimización
all_paths = []

# Parámetros
learning_rate = 0.05
momentum = 0.0
steps = 100

for start_point in start_points:
    x = start_point.reshape(1, -1)  # Asegura la forma (1, 2)
    optimizer = SGD(lr=learning_rate, momentum=momentum)
    path = [x.copy()]

    for i in range(steps):
        gradient = grad_f(x)
        x = optimizer.update(x, gradient)
        path.append(x.copy())

    all_paths.append(np.array(path).squeeze())

plot_2d_function(all_paths, f, title="SGD desde Múltiples Puntos de Partida")

In [ ]:
# Lista para almacenar las trayectorias de cada optimización
all_paths = []

# Parámetros
learning_rate = 0.05
momentum = 0.0
steps = 100
noise = 3.0

for start_point in start_points:
    x = start_point.reshape(1, -1)  # Asegura la forma (1, 2)
    optimizer = SGD(lr=learning_rate, momentum=momentum)
    path = [x.copy()]

    for i in range(steps):
        gradient = grad_noisy_f(x, noise_scale=noise)
        x = optimizer.update(x, gradient)
        path.append(x.copy())

    all_paths.append(np.array(path).squeeze())

plot_2d_function(all_paths, f, title="SGD desde Múltiples Puntos de Partida con ruido")

## SGD w/Momentum

In [ ]:
# Lista para almacenar las trayectorias de cada optimización
all_paths = []

# Parámetros
learning_rate = 0.05
momentum = 0.2
steps = 100
noise = 3.0

for start_point in start_points:
    x = start_point.reshape(1, -1)  # Asegura la forma (1, 2)
    optimizer = SGD(lr=learning_rate, momentum=momentum)
    path = [x.copy()]

    for i in range(steps):
        gradient = grad_noisy_f(x, noise_scale=noise)
        x = optimizer.update(x, gradient)
        path.append(x.copy())

    all_paths.append(np.array(path).squeeze())

plot_2d_function(all_paths, f, title="SGD desde Múltiples Puntos de Partida con ruido")

### Adam

In [ ]:
# Lista para almacenar las trayectorias de cada optimización
all_paths = []

# Parámetros
learning_rate = 0.05
#momentum = 0.2
steps = 200 # Necesitamos más steps
noise = 3.0

for start_point in start_points:
    x = start_point.reshape(1, -1)  # Asegura la forma (1, 2)
    optimizer = Adam(lr=learning_rate)
    path = [x.copy()]

    for i in range(steps):
        gradient = grad_noisy_f(x, noise_scale=noise)
        x = optimizer.update(x, gradient)
        path.append(x.copy())

    all_paths.append(np.array(path).squeeze())

plot_2d_function(all_paths, f, title="Adam desde Múltiples Puntos de Partida con ruido")